In [5]:
import pandas as pd
import glob

In [7]:
# Data Quality Validation
files = glob.glob("data/raw/games/games_*.csv")

results = []

# For each data file of games, we will check various data points for the file
for file in files:

    games = pd.read_csv(file)

    fbs_games = games[
        (games["homeClassification"] == "fbs") | (games["awayClassification"] == "fbs")
    ].copy()

    results.append({
        "season": games["season"].iloc[0],
        "all_games": len(games),
        "fbs_games": len(fbs_games),
        "fbs_vs_fbs": (
            (fbs_games["homeClassification"] == "fbs") & (fbs_games["awayClassification"] == "fbs")
        ).sum(),
        "fbs_vs_fcs": (
            (
                (fbs_games["homeClassification"] == "fbs") & (fbs_games["awayClassification"] == "fcs")
            ) |
            (
                (fbs_games["homeClassification"] == "fcs") & (fbs_games["awayClassification"] == "fbs")
            )
        ).sum(),
        "missing_home_class": games["homeClassification"].isna().sum(),
        "missing_away_class": games["awayClassification"].isna().sum(),
        "duplicate_ids": games["id"].duplicated().sum(),
        "missing_home_score": games["homePoints"].isna().sum(),
        "missing_away_score": games["awayPoints"].isna().sum()
    })

# Report the results
audit = pd.DataFrame(results).sort_values("season")

print(audit.to_string(index = False))

 season  all_games  fbs_games  fbs_vs_fbs  fbs_vs_fcs  missing_home_class  missing_away_class  duplicate_ids  missing_home_score  missing_away_score
   2015       1491        829         724         105                   0                  12              0                   0                   0
   2016       1502        832         719         113                   1                  11              0                   0                   0
   2017       1505        834         736          98                   1                  10              0                   0                   0
   2018       1511        845         733         112                   1                  14              0                   0                   0
   2019       1577        848         734         114                   0                  23              0                   0                   0
   2020        563        542         508          34                   0                   0             

In [9]:
# The quality check showed 8 games with missing scores for one or both teams. Investigate those games
files = glob.glob("data/raw/games/games_*.csv")

for file in files:

    games = pd.read_csv(file)

    missing_scores = games[
        games["homePoints"].isna() |
        games["awayPoints"].isna()
    ]

    if len(missing_scores) > 0:

        print(f"\n{'=' * 60}")
        print(f"SEASON: {games['season'].iloc[0]}")
        print(f"{'=' * 60}")

        print(
            missing_scores[
                [
                    "id",
                    "season",
                    "week",
                    "seasonType",
                    "completed",
                    "startDate",
                    "homeTeam",
                    "homeClassification",
                    "awayTeam",
                    "awayClassification",
                    "homePoints",
                    "awayPoints",
                    "notes"
                ]
            ].to_string(index=False)
        )


SEASON: 2023
       id  season  week seasonType  completed                startDate        homeTeam homeClassification         awayTeam awayClassification  homePoints  awayPoints notes
401552878    2023     9    regular      False 2023-10-28T17:00:00.000Z           Colby                iii       Middlebury                iii         NaN         NaN   NaN
401550299    2023     9    regular      False 2023-10-28T17:00:00.000Z           Bates                iii         Williams                iii         NaN         NaN   NaN
401549719    2023     9    regular      False 2023-10-28T17:00:00.000Z         Bowdoin                iii     Trinity (CT)                iii         NaN         NaN   NaN
401552884    2023    11    regular      False 2023-11-12T17:00:00.000Z Worcester State                iii Framingham State                iii         NaN         NaN   NaN

SEASON: 2024
       id  season  week seasonType  completed                startDate  homeTeam homeClassification         away

In [ ]:
# After this data check, the master data file for games will follow the following 3 guidlines:
# 1. Keep only games involving one or more FBS teams
# 2. Keep only completed games
# 3. Require a final score for both the home and away team in order to pass through to the final dataset

In [13]:
# Moving on to stats data, checking information about the source prior to cleaning
df = pd.read_csv("data/raw/stats/team_stats_2025.csv")

print(df.shape)
print(df.head())

print("\nUnique teams:", df["team"].nunique())
print("\nUnique statistics:", df["statName"].nunique())

print("\nStatistics:")
print(df["statName"].unique())

print("\nMissing values:")
print(df.isna().sum())

print("\nRecords per team:")
print(df.groupby("team").size().describe())

(8568, 5)
   season       team     conference                       statName  statValue
0    2025  Air Force  Mountain West                     firstDowns        264
1    2025  Air Force  Mountain West             firstDownsOpponent        247
2    2025  Air Force  Mountain West          fourthDownConversions         24
3    2025  Air Force  Mountain West  fourthDownConversionsOpponent         10
4    2025  Air Force  Mountain West                    fourthDowns         34

Unique teams: 136

Unique statistics: 63

Statistics:
<StringArray>
[                   'firstDowns',            'firstDownsOpponent',
         'fourthDownConversions', 'fourthDownConversionsOpponent',
                   'fourthDowns',           'fourthDownsOpponent',
                   'fumblesLost',           'fumblesLostOpponent',
              'fumblesRecovered',      'fumblesRecoveredOpponent',
                         'games',                 'interceptions',
         'interceptionsOpponent',               'in

In [15]:
# Inspect game-level stats
game_stats = pd.read_csv(
    "data/raw/game_stats/game_stats_2025.csv"
)

print(game_stats.shape)
print(game_stats.columns.tolist())
print(game_stats.head())
print(game_stats.dtypes)

print(game_stats.isna().sum())

print(game_stats["gameId"].nunique())

(3216, 8)
['gameId', 'season', 'seasonType', 'week', 'team', 'opponent', 'offense', 'defense']
      gameId  season seasonType  week           team       opponent  \
0  401752665    2025    regular     1        Alabama  Florida State   
1  401752665    2025    regular     1  Florida State        Alabama   
2  401752666    2025    regular     1    Alabama A&M       Arkansas   
3  401752666    2025    regular     1       Arkansas    Alabama A&M   
4  401752667    2025    regular     1         Auburn         Baylor   

                                             offense  \
0  {'plays': 72, 'drives': 10, 'ppa': 0.100224770...   
1  {'plays': 63, 'drives': 10, 'ppa': 0.290056628...   
2  {'plays': 55, 'drives': 12, 'ppa': 0.050584981...   
3  {'plays': 75, 'drives': 13, 'ppa': 0.407195495...   
4  {'plays': 70, 'drives': 9, 'ppa': 0.2888159132...   

                                             defense  
0  {'plays': 63, 'drives': 10, 'ppa': 0.290056628...  
1  {'plays': 72, 'drives': 10, 

In [16]:
import pandas as pd
import ast

game_stats = pd.read_csv(
    "data/raw/game_stats/game_stats_2025.csv"
)

offense = ast.literal_eval(game_stats.loc[0, "offense"])
defense = ast.literal_eval(game_stats.loc[0, "defense"])

print("OFFENSE:")
print(offense.keys())

print("\nDEFENSE:")
print(defense.keys())

print("\nOFFENSE VALUES:")
print(offense)

print("\nDEFENSE VALUES:")
print(defense)

OFFENSE:
dict_keys(['plays', 'drives', 'ppa', 'totalPPA', 'successRate', 'explosiveness', 'powerSuccess', 'stuffRate', 'lineYards', 'lineYardsTotal', 'secondLevelYards', 'secondLevelYardsTotal', 'openFieldYards', 'openFieldYardsTotal', 'standardDowns', 'passingDowns', 'rushingPlays', 'passingPlays'])

DEFENSE:
dict_keys(['plays', 'drives', 'ppa', 'totalPPA', 'successRate', 'explosiveness', 'powerSuccess', 'stuffRate', 'lineYards', 'lineYardsTotal', 'secondLevelYards', 'secondLevelYardsTotal', 'openFieldYards', 'openFieldYardsTotal', 'standardDowns', 'passingDowns', 'rushingPlays', 'passingPlays'])

OFFENSE VALUES:
{'plays': 72, 'drives': 10, 'ppa': 0.10022477071900321, 'totalPPA': 7.216183491768231, 'successRate': 0.375, 'explosiveness': 1.366452788630703, 'powerSuccess': 0.6666666666666666, 'stuffRate': 0.18518518518518517, 'lineYards': 3.3296296296296295, 'lineYardsTotal': 90, 'secondLevelYards': 0.8888888888888888, 'secondLevelYardsTotal': 24, 'openFieldYards': 0.2222222222222222, '

In [20]:
# Check week 0 games
games_2025 = pd.read_csv("data/raw/games/games_2025.csv")

stats_2025 = pd.read_csv("data/raw/game_team_stats/game_team_stats_2025.csv")

print(games_2025.shape)
print(stats_2025.shape)

print(stats_2025["gameId"].isin(games_2025["id"]).value_counts())

print(games_2025[
    ["id", "season", "week", "startDate", "homeTeam", "awayTeam"]
].sort_values("startDate").head(15))

print(games_2025[
    (games_2025["homeTeam"] == "Kansas State") |
    (games_2025["awayTeam"] == "Kansas State")
][["id", "week", "startDate", "homeTeam", "awayTeam"]].sort_values("startDate"))

(3745, 34)
(3250, 43)
gameId
True    3250
Name: count, dtype: int64
           id  season  week                 startDate              homeTeam  \
0   401756846    2025     1  2025-08-23T16:00:00.000Z          Kansas State   
1   401767476    2025     1  2025-08-23T17:00:00.000Z              Nicholls   
2   401760371    2025     1  2025-08-23T20:00:00.000Z                  UNLV   
3   401767126    2025     1  2025-08-23T20:30:00.000Z        Portland State   
4   401756847    2025     1  2025-08-23T22:30:00.000Z                Kansas   
5   401757218    2025     1  2025-08-23T23:00:00.000Z      Western Kentucky   
6   401754516    2025     1  2025-08-23T23:30:00.000Z               Hawai'i   
7   401767410    2025     1  2025-08-23T23:30:00.000Z              Southern   
8   401773590    2025     1  2025-08-28T20:00:00.000Z    Central Washington   
9   401762522    2025     1  2025-08-28T21:30:00.000Z         South Florida   
19  401773703    2025     1  2025-08-28T22:00:00.000Z          

In [26]:
import pandas as pd

master_games = pd.read_csv("data/master/master_game_data.csv")

master_cols = set(master_games.columns)

for year in range(2015, 2026):
    path = f"data/processed/game_team_stats/game_team_stats_{year}.csv"
    games = pd.read_csv(path)

    year_cols = set(games.columns)

    only_in_year = sorted(year_cols - master_cols)
    only_in_master = sorted(master_cols - year_cols)

    print(f"\n{'=' * 60}")
    print(f"{year}")
    print(f"{'=' * 60}")

    print(f"Year columns:    {len(year_cols)}")
    print(f"Master columns:  {len(master_cols)}")

    if only_in_year:
        print(f"\nColumns in {year} but NOT master ({len(only_in_year)}):")
        for col in only_in_year:
            print(f"  {col}")

    if only_in_master:
        print(f"\nColumns in master but NOT {year} ({len(only_in_master)}):")
        for col in only_in_master:
            print(f"  {col}")

    if not only_in_year and not only_in_master:
        print("\n✓ Column sets match")


2015
Year columns:    279
Master columns:  279

✓ Column sets match

2016
Year columns:    279
Master columns:  279

✓ Column sets match

2017
Year columns:    279
Master columns:  279

✓ Column sets match

2018
Year columns:    279
Master columns:  279

✓ Column sets match

2019
Year columns:    279
Master columns:  279

✓ Column sets match

2020
Year columns:    279
Master columns:  279

✓ Column sets match

2021
Year columns:    279
Master columns:  279

✓ Column sets match

2022
Year columns:    279
Master columns:  279

✓ Column sets match

2023
Year columns:    279
Master columns:  279

✓ Column sets match

2024
Year columns:    279
Master columns:  279

✓ Column sets match

2025
Year columns:    279
Master columns:  279

✓ Column sets match


In [27]:
print("Master shape:", master_games.shape)

print("\nRows by season:")
print(master_games["season"].value_counts().sort_index())

print("\nUnique games by season:")
print(
    master_games.groupby("season")["gameId"]
    .nunique()
    .sort_index()
)

print("\nDuplicate game/team rows:")
print(
    master_games.duplicated(subset=["gameId", "team"]).sum()
)

Master shape: (23622, 279)

Rows by season:
season
2015    1658
2016    1662
2017    1668
2018    1690
2019    1696
2020    1046
2021    1698
2022    3064
2023    2980
2024    3210
2025    3250
Name: count, dtype: int64

Unique games by season:
season
2015     829
2016     831
2017     834
2018     845
2019     848
2020     523
2021     849
2022    1532
2023    1490
2024    1605
2025    1625
Name: gameId, dtype: int64

Duplicate game/team rows:
0


In [29]:
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

MASTER_PATH = "data/master/master_game_data.csv"

# Change this if your transformed team-level files are stored elsewhere
TEAM_STATS_PATH = "data/processed/game_team_stats/game_team_stats_2025.csv"


# ============================================================
# LOAD DATA
# ============================================================

print("Loading master game-level data...")
df = pd.read_csv(MASTER_PATH)

print(f"Master shape: {df.shape}")

print("\nLoading team-level data...")
team_df = pd.read_csv(TEAM_STATS_PATH)

print(f"Team-level shape: {team_df.shape}")


# ============================================================
# BASIC INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("BASIC DATA INFORMATION")
print("=" * 70)

print("\nMaster columns:")
print(len(df.columns))

print("\nTeam-level columns:")
print(len(team_df.columns))

print("\nMaster seasons:")
print(sorted(df["season"].unique()))

print("\nTeam-level seasons:")
print(sorted(team_df["season"].unique()))


# ============================================================
# CHECK 1: CHRONOLOGICAL ORDER
# ============================================================

print("\n" + "=" * 70)
print("CHECK 1: CHRONOLOGICAL ORDER")
print("=" * 70)

team_df["startDate"] = pd.to_datetime(team_df["startDate"])

team_df = team_df.sort_values(
    ["season", "team", "startDate"]
).reset_index(drop=True)

# Check whether each team's games are chronologically ordered
chronology_check = (
    team_df
    .groupby(["season", "team"])["startDate"]
    .apply(lambda x: x.is_monotonic_increasing)
)

chronology_errors = chronology_check[~chronology_check]

print(f"Teams with chronology errors: {len(chronology_errors)}")

if len(chronology_errors) == 0:
    print("✓ All team games are chronologically ordered")
else:
    print("⚠ Chronology problems found:")
    print(chronology_errors)


# ============================================================
# CHECK 2: GAMES BEFORE
# ============================================================

print("\n" + "=" * 70)
print("CHECK 2: GAMES BEFORE")
print("=" * 70)

# Calculate the expected number of games played BEFORE each game
team_df["expected_gamesBefore"] = (
    team_df
    .groupby(["season", "team"])
    .cumcount()
)

# Check whether the expected column exists
if "gamesBefore" in team_df.columns:

    mismatches = team_df[
        team_df["gamesBefore"] != team_df["expected_gamesBefore"]
    ]

    print(f"gamesBefore mismatches: {len(mismatches)}")

    if len(mismatches) == 0:
        print("✓ gamesBefore is correct")
    else:
        print("\n⚠ gamesBefore mismatches:")
        print(
            mismatches[
                [
                    "season",
                    "team",
                    "gameId",
                    "week",
                    "startDate",
                    "gamesBefore",
                    "expected_gamesBefore",
                ]
            ].head(25)
        )

else:
    print("⚠ Column 'gamesBefore' not found")


# ============================================================
# CHECK 3: WINS BEFORE
# ============================================================

print("\n" + "=" * 70)
print("CHECK 3: WINS BEFORE")
print("=" * 70)

# Identify the team's game result
#
# Home team:
#   homePoints > awayPoints -> win
#
# Away team:
#   awayPoints > homePoints -> win
#
# We need to create this from the game-level information
# available in team_df.

if {
    "homePoints",
    "awayPoints",
    "homeAway"
}.issubset(team_df.columns):

    team_df["win_current"] = np.where(
        (
            (team_df["homeAway"] == "home") &
            (team_df["homePoints"] > team_df["awayPoints"])
        )
        |
        (
            (team_df["homeAway"] == "away") &
            (team_df["awayPoints"] > team_df["homePoints"])
        ),
        1,
        0
    )

    # Calculate wins BEFORE the current game
    team_df["expected_winsBefore"] = (
        team_df
        .groupby(["season", "team"])["win_current"]
        .transform(lambda x: x.shift(1).fillna(0).cumsum())
    )

    if "winsBefore" in team_df.columns:

        # Only compare games with known scores
        valid = (
            team_df["homePoints"].notna()
            & team_df["awayPoints"].notna()
        )

        mismatches = team_df[
            valid
            & (
                team_df["winsBefore"]
                != team_df["expected_winsBefore"]
            )
        ]

        print(f"winsBefore mismatches: {len(mismatches)}")

        if len(mismatches) == 0:
            print("✓ winsBefore is correct")
        else:
            print("\n⚠ winsBefore mismatches:")
            print(
                mismatches[
                    [
                        "season",
                        "team",
                        "gameId",
                        "week",
                        "winsBefore",
                        "expected_winsBefore",
                    ]
                ].head(25)
            )

    else:
        print("⚠ Column 'winsBefore' not found")

else:
    print(
        "⚠ Could not calculate winsBefore because "
        "required columns are missing."
    )


# ============================================================
# CHECK 4: LOSSES BEFORE
# ============================================================

print("\n" + "=" * 70)
print("CHECK 4: LOSSES BEFORE")
print("=" * 70)

if "win_current" in team_df.columns:

    team_df["loss_current"] = np.where(
        (
            team_df["homePoints"].notna()
            & team_df["awayPoints"].notna()
            & (team_df["win_current"] == 0)
        ),
        1,
        0
    )

    team_df["expected_lossesBefore"] = (
        team_df
        .groupby(["season", "team"])["loss_current"]
        .transform(lambda x: x.shift(1).fillna(0).cumsum())
    )

    if "lossesBefore" in team_df.columns:

        valid = (
            team_df["homePoints"].notna()
            & team_df["awayPoints"].notna()
        )

        mismatches = team_df[
            valid
            & (
                team_df["lossesBefore"]
                != team_df["expected_lossesBefore"]
            )
        ]

        print(f"lossesBefore mismatches: {len(mismatches)}")

        if len(mismatches) == 0:
            print("✓ lossesBefore is correct")
        else:
            print("\n⚠ lossesBefore mismatches:")
            print(
                mismatches[
                    [
                        "season",
                        "team",
                        "gameId",
                        "week",
                        "lossesBefore",
                        "expected_lossesBefore",
                    ]
                ].head(25)
            )

    else:
        print("⚠ Column 'lossesBefore' not found")


# ============================================================
# CHECK 5: WIN PERCENTAGE BEFORE
# ============================================================

print("\n" + "=" * 70)
print("CHECK 5: WIN PERCENTAGE BEFORE")
print("=" * 70)

if {
    "expected_winsBefore",
    "expected_gamesBefore"
}.issubset(team_df.columns):

    team_df["expected_winPctBefore"] = np.where(
        team_df["expected_gamesBefore"] > 0,
        team_df["expected_winsBefore"]
        / team_df["expected_gamesBefore"],
        0
    )

    if "winPctBefore" in team_df.columns:

        mismatches = team_df[
            ~np.isclose(
                team_df["winPctBefore"],
                team_df["expected_winPctBefore"],
                equal_nan=True
            )
        ]

        print(f"winPctBefore mismatches: {len(mismatches)}")

        if len(mismatches) == 0:
            print("✓ winPctBefore is correct")
        else:
            print("\n⚠ winPctBefore mismatches:")
            print(
                mismatches[
                    [
                        "season",
                        "team",
                        "gameId",
                        "week",
                        "winPctBefore",
                        "expected_winPctBefore",
                    ]
                ].head(25)
            )


# ============================================================
# CHECK 6: SUSPICIOUS TARGET / POST-GAME FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CHECK 6: POTENTIAL TARGET / POST-GAME FEATURES")
print("=" * 70)

suspicious_keywords = [
    "score",
    "points",
    "margin",
    "winner",
    "result",
]

suspicious_columns = []

for col in df.columns:

    col_lower = col.lower()

    if any(keyword in col_lower for keyword in suspicious_keywords):
        suspicious_columns.append(col)

print(f"\nPotentially suspicious columns: {len(suspicious_columns)}")

for col in suspicious_columns:
    print(f"  {col}")


# ============================================================
# CHECK 7: CURRENT-GAME STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("CHECK 7: POTENTIAL CURRENT-GAME STATISTICS")
print("=" * 70)

stat_keywords = [
    "ppa",
    "yards",
    "plays",
    "drives",
    "successrate",
    "explosiveness",
    "powerSuccess",
    "stuffRate",
    "lineYards",
    "passingPlays",
    "rushingPlays",
]

potential_game_stats = []

for col in df.columns:

    col_lower = col.lower()

    if any(keyword.lower() in col_lower for keyword in stat_keywords):
        potential_game_stats.append(col)

print(
    f"\nPotential game-stat columns: "
    f"{len(potential_game_stats)}"
)

for col in potential_game_stats:
    print(f"  {col}")


# ============================================================
# CHECK 8: LOOK FOR NON-PREGAME FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CHECK 8: FEATURES WITHOUT 'BEFORE' / 'PREGAME'")
print("=" * 70)

identifier_columns = {
    "id",
    "gameId",
    "season",
    "week",
    "seasonType",
    "completed",
    "startDate",
    "homeTeam",
    "awayTeam",
    "homeClassification",
    "awayClassification",
    "homeAway",
    "team",
    "opponent",
}

non_pregame_features = []

for col in df.columns:

    if col in identifier_columns:
        continue

    col_lower = col.lower()

    if (
        "before" not in col_lower
        and "pregame" not in col_lower
    ):
        non_pregame_features.append(col)

print(
    f"\nFeatures without 'Before' or 'Pregame': "
    f"{len(non_pregame_features)}"
)

for col in non_pregame_features:
    print(f"  {col}")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("LEAKAGE CHECK COMPLETE")
print("=" * 70)

print("""
Review the results above carefully.

The most important checks are:

1. Chronological ordering
2. gamesBefore
3. winsBefore
4. lossesBefore
5. winPctBefore
6. Suspicious post-game/target columns
7. Current-game statistics
8. Features that aren't explicitly identified as pregame

A clean result on the first five checks is strong evidence that
your cumulative pregame statistics are being calculated correctly.

The remaining feature columns need to be reviewed individually
to determine whether they are legitimate pregame features.
""")

Loading master game-level data...
Master shape: (23622, 279)

Loading team-level data...
Team-level shape: (3250, 279)

BASIC DATA INFORMATION

Master columns:
279

Team-level columns:
279

Master seasons:
[np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Team-level seasons:
[np.int64(2025)]

CHECK 1: CHRONOLOGICAL ORDER
Teams with chronology errors: 0
✓ All team games are chronologically ordered

CHECK 2: GAMES BEFORE
gamesBefore mismatches: 0
✓ gamesBefore is correct

CHECK 3: WINS BEFORE


C:\Users\colby\AppData\Local\Temp\ipykernel_3016\2625087480.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_df["expected_gamesBefore"] = (
C:\Users\colby\AppData\Local\Temp\ipykernel_3016\2625087480.py:154: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_df["win_current"] = np.where(
C:\Users\colby\AppData\Local\Temp\ipykernel_3016\2625087480.py:169: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all 

winsBefore mismatches: 0
✓ winsBefore is correct

CHECK 4: LOSSES BEFORE
⚠ Column 'lossesBefore' not found

CHECK 5: WIN PERCENTAGE BEFORE
winPctBefore mismatches: 315

⚠ winPctBefore mismatches:
     season                    team     gameId  week  winPctBefore  \
0      2025       Abilene Christian  401762457     1           NaN   
14     2025                  Adrian  401767656     2           NaN   
15     2025               Air Force  401760359     1           NaN   
27     2025                   Akron  401762789     1           NaN   
39     2025                 Alabama  401752665     1           NaN   
52     2025             Alabama A&M  401752666     1           NaN   
64     2025           Alabama State  401762430     1           NaN   
75     2025            Albany State  401769832     3           NaN   
76     2025            Alcorn State  401767369     1           NaN   
88     2025                   Allen  401767634     1           NaN   
89     2025  American Internationa

C:\Users\colby\AppData\Local\Temp\ipykernel_3016\2625087480.py:240: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_df["expected_lossesBefore"] = (
C:\Users\colby\AppData\Local\Temp\ipykernel_3016\2625087480.py:297: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_df["expected_winPctBefore"] = np.where(


In [31]:
import pandas as pd

df = pd.read_csv("data/raw/game_stats/game_stats_2025.csv")

print(df.columns.tolist())

['gameId', 'season', 'seasonType', 'week', 'team', 'opponent', 'offense', 'defense']


In [32]:
import pandas as pd

for year in range(2015, 2026):

    game_features = pd.read_csv(
        f"data/processed/game_features/game_features_{year}.csv"
    )

    advanced_features = pd.read_csv(
        f"data/processed/features/features_{year}.csv"
    )

    print("\n" + "=" * 60)
    print(year)
    print("=" * 60)

    print("\nGame-level features ID columns:")
    print([
        col for col in game_features.columns
        if "game" in col.lower() or col == "id"
    ])

    print("\nAdvanced features ID columns:")
    print([
        col for col in advanced_features.columns
        if "game" in col.lower() or col == "id"
    ])


2015

Game-level features ID columns:
['gameId', 'gamesBefore_home', 'homeGamesBefore_home', 'gamesBefore_away', 'awayGamesBefore_away']

Advanced features ID columns:
['id', 'conferenceGame', 'homePostgameWinProbability', 'homePregameElo', 'homePostgameElo', 'awayPostgameWinProbability', 'awayPregameElo', 'awayPostgameElo', 'home_pregame_offense_ppa', 'home_pregame_offense_successRate', 'home_pregame_offense_explosiveness', 'home_pregame_defense_ppa', 'home_pregame_defense_successRate', 'home_pregame_defense_explosiveness', 'away_pregame_offense_ppa', 'away_pregame_offense_successRate', 'away_pregame_offense_explosiveness', 'away_pregame_defense_ppa', 'away_pregame_defense_successRate', 'away_pregame_defense_explosiveness']

2016

Game-level features ID columns:
['gameId', 'gamesBefore_home', 'homeGamesBefore_home', 'gamesBefore_away', 'awayGamesBefore_away']

Advanced features ID columns:
['id', 'conferenceGame', 'homePostgameWinProbability', 'homePregameElo', 'homePostgameElo', 'aw

In [33]:
import pandas as pd
from pathlib import Path


YEARS = range(2015, 2026)

RAW_GAMES_DIR = Path("data/raw/games")
ADVANCED_DIR = Path("data/processed/features")
GAME_FEATURES_DIR = Path("data/processed/game_features")


def validate_year(year):
    print("\n" + "=" * 70)
    print(f"VALIDATING MISSING GAME-LEVEL FEATURES: {year}")
    print("=" * 70)

    raw_path = RAW_GAMES_DIR / f"games_{year}.csv"
    advanced_path = ADVANCED_DIR / f"features_{year}.csv"
    game_features_path = GAME_FEATURES_DIR / f"game_features_{year}.csv"

    raw = pd.read_csv(raw_path)
    advanced = pd.read_csv(advanced_path)
    game_features = pd.read_csv(game_features_path)

    print(f"Raw games:          {len(raw)}")
    print(f"Advanced statistics:{len(advanced)}")
    print(f"Game-level features:{len(game_features)}")

    # ------------------------------------------------------------
    # Identify FBS games from raw games
    # Option B = keep every game involving at least one FBS team
    # ------------------------------------------------------------

    fbs_games = raw[
        (raw["homeClassification"] == "fbs") |
        (raw["awayClassification"] == "fbs")
    ].copy()

    print(f"\nOption B games in raw data: {len(fbs_games)}")

    # ------------------------------------------------------------
    # IDs
    # ------------------------------------------------------------

    raw_ids = set(fbs_games["id"].astype(int))

    advanced_ids = set(advanced["id"].astype(int))

    game_feature_ids = set(game_features["gameId"].astype(int))

    # ------------------------------------------------------------
    # Missing game-level features
    # ------------------------------------------------------------

    missing_game_features = sorted(
        raw_ids - game_feature_ids
    )

    print(
        f"Missing game-level features: "
        f"{len(missing_game_features)}"
    )

    if not missing_game_features:
        print("\nPASS: No Option B games are missing game-level features.")
        return True

    # ------------------------------------------------------------
    # Build diagnostic table
    # ------------------------------------------------------------

    missing = fbs_games[
        fbs_games["id"].astype(int).isin(missing_game_features)
    ].copy()

    missing["has_advanced_stats"] = (
        missing["id"].astype(int).isin(advanced_ids)
    )

    # ------------------------------------------------------------
    # Display missing games
    # ------------------------------------------------------------

    columns = [
        "id",
        "week",
        "seasonType",
        "homeTeam",
        "homeClassification",
        "awayTeam",
        "awayClassification",
        "has_advanced_stats",
    ]

    print("\nMissing games:")
    print(
        missing[columns]
        .sort_values(["week", "homeTeam"])
        .to_string(index=False)
    )

    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------

    advanced_missing = missing[
        ~missing["has_advanced_stats"]
    ]

    advanced_present = missing[
        missing["has_advanced_stats"]
    ]

    print("\n" + "-" * 70)
    print("DIAGNOSIS")
    print("-" * 70)

    print(
        f"Missing game-level + missing advanced stats: "
        f"{len(advanced_missing)}"
    )

    print(
        f"Missing game-level + HAS advanced stats: "
        f"{len(advanced_present)}"
    )

    if len(advanced_present) > 0:
        print(
            "\nIMPORTANT:"
            "\nThese games have advanced statistics but no game-level "
            "features."
            "\nThis indicates the problem is in the game-level feature "
            "creation pipeline."
        )

    if len(advanced_missing) > 0:
        print(
            "\nThese games are missing both game-level features and "
            "advanced statistics."
        )

    return False


def main():
    results = {}

    for year in YEARS:
        try:
            results[year] = validate_year(year)

        except Exception as e:
            print(f"\nERROR processing {year}:")
            print(f"{type(e).__name__}: {e}")
            results[year] = False

    print("\n\n" + "=" * 70)
    print("FINAL MISSING GAME-LEVEL FEATURE VALIDATION")
    print("=" * 70)

    for year, passed in results.items():
        print(f"{year}: {'PASS' if passed else 'FAIL'}")


if __name__ == "__main__":
    main()


VALIDATING MISSING GAME-LEVEL FEATURES: 2015
Raw games:          1491
Advanced statistics:829
Game-level features:829

Option B games in raw data: 829
Missing game-level features: 0

PASS: No Option B games are missing game-level features.

VALIDATING MISSING GAME-LEVEL FEATURES: 2016
Raw games:          1502
Advanced statistics:832
Game-level features:831

Option B games in raw data: 832
Missing game-level features: 1

Missing games:
       id  week seasonType homeTeam homeClassification awayTeam awayClassification  has_advanced_stats
400868914     6    regular     Duke                fbs     Army                fbs                True

----------------------------------------------------------------------
DIAGNOSIS
----------------------------------------------------------------------
Missing game-level + missing advanced stats: 0
Missing game-level + HAS advanced stats: 1

IMPORTANT:
These games have advanced statistics but no game-level features.
This indicates the problem is in t

In [1]:
import pandas as pd

df = pd.read_csv("data/processed/modeling/logistic_regression_data.csv")

print(df.columns.to_list())

['season', 'gameId', 'awayCompletionPctBefore_away', 'awayCompletionsAvgBefore_away', 'awayCompletionsBefore_away', 'awayDefensiveTDsAvgBefore_away', 'awayDefensiveTDsBefore_away', 'awayFirstDownsAvgBefore_away', 'awayFirstDownsBefore_away', 'awayFourthDownAttemptsAvgBefore_away', 'awayFourthDownAttemptsBefore_away', 'awayFourthDownConversionsAvgBefore_away', 'awayFourthDownConversionsBefore_away', 'awayFourthDownPctBefore_away', 'awayFumblesLostAvgBefore_away', 'awayFumblesLostBefore_away', 'awayGamesBefore_away', 'awayInterceptionsAvgBefore_away', 'awayInterceptionsBefore_away', 'awayNetPassingYardsAvgBefore_away', 'awayNetPassingYardsBefore_away', 'awayPassAttemptsAvgBefore_away', 'awayPassAttemptsBefore_away', 'awayPassesDeflectedAvgBefore_away', 'awayPassesDeflectedBefore_away', 'awayPassingTDsAvgBefore_away', 'awayPassingTDsBefore_away', 'awayPenaltiesAvgBefore_away', 'awayPenaltiesBefore_away', 'awayPenaltyYardsAvgBefore_away', 'awayPenaltyYardsBefore_away', 'awayPointDifferenti

In [4]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent

filepath = ROOT / "data/raw/player/recruiting/player_recruiting_2025.csv"

df = pd.read_csv(filepath)

print("=" * 80)
print("2025 RECRUITING DATA")
print("=" * 80)

print(f"\nRows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

print("\nColumns:")
for col in df.columns:
    print(f"  {col}")

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst 10 rows:")
print(df.head(10).to_string(index=False))

2025 RECRUITING DATA

Rows: 4,120
Columns: 17

Columns:
  id
  athleteId
  recruitType
  year
  ranking
  name
  school
  committedTo
  position
  height
  weight
  stars
  rating
  city
  stateProvince
  country
  hometownInfo

Data types:
id                 int64
athleteId        float64
recruitType          str
year               int64
ranking          float64
name                 str
school               str
committedTo          str
position             str
height           float64
weight           float64
stars            float64
rating           float64
city                 str
stateProvince        str
country              str
hometownInfo         str
dtype: object

Missing values:
id                 0
athleteId        867
recruitType        0
year               0
ranking          477
name               0
school             0
committedTo      473
position           0
height             1
weight             1
stars            476
rating           476
city               0
stateProv

In [5]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent

filepath = ROOT / "data/raw/player/recruiting/player_recruiting_2025.csv"

df = pd.read_csv(filepath)

print("=" * 80)
print("2025 RECRUITING DATA AUDIT")
print("=" * 80)

# ---------------------------------------------------------------------
# BASIC STRUCTURE
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("1. BASIC STRUCTURE")
print("=" * 80)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

print("\nUnique athlete IDs:")
print(f"  Non-missing: {df['athleteId'].notna().sum():,}")
print(f"  Unique: {df['athleteId'].nunique():,}")
print(f"  Duplicate athlete IDs: {df['athleteId'].duplicated().sum():,}")


# ---------------------------------------------------------------------
# RECRUIT TYPE
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("2. RECRUIT TYPE")
print("=" * 80)

print(df["recruitType"].value_counts(dropna=False).to_string())

print("\nRecruit type percentages:")
print(
    (df["recruitType"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
    .to_string()
)


# ---------------------------------------------------------------------
# COMMITMENT
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("3. COMMITMENT STATUS")
print("=" * 80)

uncommitted = df["committedTo"].isna()

print(f"Committed:   {(~uncommitted).sum():,}")
print(f"Uncommitted: {uncommitted.sum():,}")

print("\nTop committed teams:")
print(
    df.loc[~uncommitted, "committedTo"]
      .value_counts()
      .head(25)
      .to_string()
)


# ---------------------------------------------------------------------
# RECRUITING METRICS
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("4. RECRUITING METRICS")
print("=" * 80)

for col in ["ranking", "stars", "rating"]:
    print(f"\n{col}:")
    print(df[col].describe().to_string())

    print(f"Missing: {df[col].isna().sum():,}")


# ---------------------------------------------------------------------
# RATING / STARS / RANKING RELATIONSHIP
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("5. RATING / STARS / RANKING RELATIONSHIP")
print("=" * 80)

print("\nStars distribution:")
print(df["stars"].value_counts(dropna=False).sort_index().to_string())

print("\nRating quantiles:")
print(
    df["rating"]
    .quantile([0, .01, .05, .10, .25, .50, .75, .90, .95, .99, 1])
    .to_string()
)

print("\nRanking quantiles:")
print(
    df["ranking"]
    .quantile([0, .01, .05, .10, .25, .50, .75, .90, .95, .99, 1])
    .to_string()
)


# ---------------------------------------------------------------------
# POSITION
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("6. POSITION")
print("=" * 80)

print(f"Unique positions: {df['position'].nunique()}")

print(
    df["position"]
    .value_counts(dropna=False)
    .to_string()
)


# ---------------------------------------------------------------------
# COMMITTED TEAM COVERAGE
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("7. COMMITTED TEAM COVERAGE")
print("=" * 80)

print(f"Unique committed teams: {df['committedTo'].nunique(dropna=True)}")

print("\nMissing committed team by recruit type:")
print(
    pd.crosstab(
        df["recruitType"],
        df["committedTo"].isna(),
        margins=True
    )
)


# ---------------------------------------------------------------------
# RECORDS WITH MISSING RECRUITING METRICS
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("8. MISSING RECRUITING METRICS")
print("=" * 80)

metric_cols = ["ranking", "stars", "rating"]

missing_metrics = (
    df[metric_cols]
    .isna()
    .sum(axis=1)
)

print(
    missing_metrics
    .value_counts()
    .sort_index()
    .rename("number_of_records")
    .to_string()
)

print("\nRecords missing all three:")
print((missing_metrics == 3).sum())


# ---------------------------------------------------------------------
# SAMPLE UNCOMMITTED RECORDS
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("9. UNCOMMITTED RECRUITS")
print("=" * 80)

print(
    df.loc[
        df["committedTo"].isna(),
        [
            "name",
            "recruitType",
            "position",
            "ranking",
            "stars",
            "rating"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ---------------------------------------------------------------------
# SAMPLE RECORDS WITH MISSING RATINGS
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("10. RECORDS WITH MISSING RATING")
print("=" * 80)

print(
    df.loc[
        df["rating"].isna(),
        [
            "name",
            "recruitType",
            "committedTo",
            "position",
            "ranking",
            "stars",
            "rating"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

2025 RECRUITING DATA AUDIT

1. BASIC STRUCTURE
Rows: 4,120
Columns: 17
Duplicate rows: 0

Unique athlete IDs:
  Non-missing: 3,253
  Unique: 3,253
  Duplicate athlete IDs: 866

2. RECRUIT TYPE
recruitType
HighSchool    4120

Recruit type percentages:
recruitType
HighSchool    100.0

3. COMMITMENT STATUS
Committed:   3,647
Uncommitted: 473

Top committed teams:
committedTo
Army                  50
Navy                  50
Air Force             49
Syracuse              35
Washington State      33
North Carolina        30
Washington            30
Rutgers               30
North Dakota State    30
Georgia               29
Penn State            29
Northern Illinois     29
UTEP                  29
Nevada                29
Ohio State            28
USC                   28
Duke                  28
Utah State            28
Grand Valley State    28
Florida               27
TCU                   27
Boston College        27
Auburn                26
Indiana               26
Boise State           26


In [9]:
# ---------------------------------------------------------------------
# 11. RECRUITING TEAM ALIGNMENT WITH FINAL FEATURES
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("11. RECRUITING TEAM ALIGNMENT")
print("=" * 80)

# Load 2025 final features
final_features_path = (
    ROOT / "data/processed/features/final/final_features_2025.csv"
)

final_features = pd.read_csv(final_features_path)

# Team names appearing in the 2025 modeling population
model_teams = set(
    pd.concat([
        final_features["homeTeam"],
        final_features["awayTeam"]
    ]).dropna().unique()
)

# Team names appearing in recruiting data
recruiting_teams = set(
    df["committedTo"].dropna().unique()
)

# Compare
overlap = model_teams & recruiting_teams
model_only = model_teams - recruiting_teams
recruiting_only = recruiting_teams - model_teams

print(f"\nFinal-feature teams:          {len(model_teams):,}")
print(f"Recruiting teams:             {len(recruiting_teams):,}")
print(f"Teams in both:                {len(overlap):,}")
print(f"Modeling teams with no match: {len(model_only):,}")
print(f"Recruiting-only teams:        {len(recruiting_only):,}")

print("\n" + "-" * 80)
print("MODELING TEAMS WITH NO RECRUITING MATCH")
print("-" * 80)

for team in sorted(model_only):
    print(team)

print("\n" + "-" * 80)
print("RECRUITING TEAMS NOT IN MODELING DATA")
print("-" * 80)

for team in sorted(recruiting_only):
    print(team)


11. RECRUITING TEAM ALIGNMENT

Final-feature teams:          230
Recruiting teams:             239
Teams in both:                206
Modeling teams with no match: 24
Recruiting-only teams:        33

--------------------------------------------------------------------------------
MODELING TEAMS WITH NO RECRUITING MATCH
--------------------------------------------------------------------------------
Abilene Christian
Arkansas-Pine Bluff
Bethune-Cookman
Central Connecticut
East Texas A&M
Elon
Florida A&M
Grambling
Long Island University
Merrimack
Monmouth
Nicholls
North Carolina A&T
SE Louisiana
Saint Francis
South Carolina State
Southern
Stony Brook
Tarleton State
Tennessee Tech
The Citadel
UAlbany
VMI
Villanova

--------------------------------------------------------------------------------
RECRUITING TEAMS NOT IN MODELING DATA
--------------------------------------------------------------------------------
Brown
Butler
Columbia
Cornell
Dartmouth
Davidson
Dayton
Drake
Georgetown
Gran

In [11]:
# ---------------------------------------------------------------------
# 13. UNMATCHED RECRUITING TEAM IMPACT
# ---------------------------------------------------------------------

print("\n" + "=" * 80)
print("13. UNMATCHED RECRUITING TEAM IMPACT")
print("=" * 80)

# Load 2025 recruiting data
recruiting_path = (
    ROOT / "data/raw/player/recruiting/player_recruiting_2025.csv"
)

df = pd.read_csv(recruiting_path)

# Load 2025 final features
final_features_path = (
    ROOT / "data/processed/features/final/final_features_2025.csv"
)

final_features = pd.read_csv(final_features_path)

# Teams appearing in final features
model_teams = set(
    pd.concat([
        final_features["homeTeam"],
        final_features["awayTeam"]
    ]).dropna().unique()
)

# Only committed recruits can be assigned to a team
committed_recruits = df[
    df["committedTo"].notna()
].copy()

# Recruits committed to teams not appearing in final features
unmatched_recruits = committed_recruits[
    ~committed_recruits["committedTo"].isin(model_teams)
].copy()

# Summarize unmatched teams
unmatched_recruit_teams = (
    unmatched_recruits
    .groupby("committedTo")
    .agg(
        recruits=("name", "count"),
        rated_recruits=("rating", "count"),
        avg_rating=("rating", "mean")
    )
    .sort_values("recruits", ascending=False)
)

print("\n" + "-" * 80)
print("UNMATCHED RECRUITING TEAMS")
print("-" * 80)

print(unmatched_recruit_teams.to_string())

print("\n" + "-" * 80)
print("SUMMARY")
print("-" * 80)

print(f"Final-feature teams:              {len(model_teams):,}")
print(f"Committed recruits:                {len(committed_recruits):,}")
print(f"Unmatched committed recruits:      {len(unmatched_recruits):,}")

print(
    f"Percentage of committed recruits unmatched: "
    f"{len(unmatched_recruits) / len(committed_recruits) * 100:.2f}%"
)


13. UNMATCHED RECRUITING TEAM IMPACT

--------------------------------------------------------------------------------
UNMATCHED RECRUITING TEAMS
--------------------------------------------------------------------------------
                        recruits  rated_recruits  avg_rating
committedTo                                                 
North Dakota State            30              23    0.817735
Grand Valley State            28              28    0.777446
South Dakota State            24              15    0.818813
Montana                       17              15    0.813333
Drake                         14               5    0.758000
Saint Francis (PA)            13               4    0.829300
West Georgia                  11               4    0.847775
Cornell                       11               8    0.813612
Pennsylvania                  11               4    0.815000
Yale                          11               8    0.810175
Harvard                       10        